# Assignment 3: Supervised Land Use Classification with Google Earth Engine

In [ ]:
!pip install earthengine-api geemap ipywidgets leafmap



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.7/518.7 kB 5.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 219.1/219.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 28.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.4/41.4 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 108.6/108.6 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.7/2.7 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 25.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 765.5/765.5 kB 27.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.2/194.2 kB 12.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.0/74.0 kB 4.7 MB/s eta 0:00:00


In [ ]:
!pip install rasterio matplotlib numpy



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.2/22.2 MB 41.7 MB/s eta 0:00:00


In [ ]:
# Import required libraries
import ee
import geemap
import ipywidgets as widgets
from IPython.display import display
import leafmap

import rasterio
import matplotlib
import numpy

import geopandas as gpd

In [ ]:
ee.Authenticate()
ee.Initialize(project='ee-hw2222')

## Download TIFF image for labelling

In [ ]:
# Define area of interest (AOI)
charleston_aoi = ee.Geometry.Rectangle([-80.30, 32.55, -79.75, 33.00])

charleston_image = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
                 .filterBounds(charleston_aoi)
                 .filterDate('2022-01-01', '2023-12-31')
                 .sort('CLOUD_COVER')
                 .first()  # Take the best one
                 .select(["SR_B1", "SR_B2", "SR_B3", "SR_B4", "SR_B5", "SR_B6", "SR_B7"])
)

## Labeling

processed in ArcGIS, labeled the different land uses, and exported to shapefile, will continue to in python to continue processing.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
labels = gpd.read_file("/content/drive/MyDrive/DeepLearning_HW2/Labels/Label_Output.shp")  # Assuming the unzipped file is named labels.shp

In [ ]:
labels.head()

print("Unique class names:", labels["Class"].unique())


Unique class names: ['Urban' 'Covered' 'Water' 'Bare']


In [ ]:
def add_spectral_indices(image):
    # Bands: B2=Blue, B3=Green, B4=Red, B5=NIR, B6=SWIR1, B7=SWIR2 in L8
    # NDVI = (NIR - Red) / (NIR + Red)
    ndvi = image.expression(
        '(NIR - RED) / (NIR + RED)',
        {
            'NIR': image.select('SR_B5'),
            'RED': image.select('SR_B4')
        }
    ).rename('NDVI')

    # NDBI = (SWIR1 - NIR) / (SWIR1 + NIR)
    ndbi = image.expression(
        '(SWIR - NIR) / (SWIR + NIR)',
        {
            'SWIR': image.select('SR_B6'),
            'NIR':  image.select('SR_B5')
        }
    ).rename('NDBI')

    # MNDWI = (Green - SWIR1) / (Green + SWIR1)
    mndwi = image.expression(
        '(GREEN - SWIR) / (GREEN + SWIR)',
        {
            'GREEN': image.select('SR_B3'),
            'SWIR':  image.select('SR_B6')
        }
    ).rename('MNDWI')

    # Add indices as bands
    return image.addBands([ndvi, ndbi, mndwi])

charleston_image = add_spectral_indices(charleston_image)


In [ ]:
def normalize_bands(image, band_names, min_dict, max_dict):
    # Normalize each band to 0-1
    for band in band_names:
        min_val = min_dict[band]
        max_val = max_dict[band]
        normalized = image.select(band).subtract(min_val).divide(max_val - min_val).rename(band + '_norm')
        image = image.addBands(normalized)
    return image

# Example min/max for each band — these are placeholders; adapt for your dataset
min_values = {
    'SR_B1': 0.0, 'SR_B2': 0.0, 'SR_B3': 0.0, 'SR_B4': 0.0, 'SR_B5': 0.0, 'SR_B6': 0.0, 'SR_B7': 0.0,
    'NDVI': -1.0, 'NDBI': -1.0, 'MNDWI': -1.0
}
max_values = {
    'SR_B1': 1.0, 'SR_B2': 1.0, 'SR_B3': 1.0, 'SR_B4': 1.0, 'SR_B5': 1.0, 'SR_B6': 1.0, 'SR_B7': 1.0,
    'NDVI': 1.0, 'NDBI': 1.0, 'MNDWI': 1.0
}

all_bands = ['SR_B1','SR_B2','SR_B3','SR_B4','SR_B5','SR_B6','SR_B7','NDVI','NDBI','MNDWI']
charleston_image = normalize_bands(charleston_image, all_bands, min_values, max_values)


In [ ]:


# Reproject to EPSG:4326 (the typical lat/lon WGS84)
labels = labels.to_crs(epsg=4326)

# A helper function to fix or remove invalid geometries
def fix_invalid_geometries(gdf):
    # 1) buffer(0) often fixes slight geometry errors
    # 2) drop empty or still-invalid geometries
    gdf = gdf.copy()
    gdf['geometry'] = gdf['geometry'].buffer(0)
    gdf = gdf[~gdf.geometry.is_empty]
    gdf = gdf[gdf.is_valid]
    return gdf

labels = fix_invalid_geometries(labels)

# OPTIONAL: If your shapefile has mixed geometry types (e.g., points and polygons),
# you may need to filter to only polygons if you plan to do polygon sampling:
# labels = labels[labels.geometry.type == 'Polygon']

# -------------------------
# 3. Convert to EE FeatureCollection
# -------------------------
label_features = geemap.geopandas_to_ee(labels, geodesic=False)

# Now label_features should be a valid Earth Engine FeatureCollection.
# You can confirm by printing or calling .size() or .first() on it:
print("Number of features:", label_features.size().getInfo())

# -------------------------
# 4. Example: Sample a Landsat Image
# -------------------------
aoi = ee.Geometry.Rectangle([-80.30, 32.55, -79.75, 33.00])

# Example: Load and select some Landsat bands
landsat = (ee.ImageCollection("LANDSAT/LC08/C02/T1_L2")
           .filterBounds(aoi)
           .filterDate('2021-01-01', '2023-12-31')
           .sort('CLOUD_COVER')
           .first()
           .select(["SR_B1","SR_B2","SR_B3","SR_B4","SR_B5","SR_B6","SR_B7"]))

# Sample the image with your label features (assuming your label column is named 'class')
training_data = landsat.sampleRegions(
    collection = label_features,
    properties = ['class'],
    scale = 30
)

# Print number of samples
print("Number of training samples:", training_data.size().getInfo())


Number of features: 0
Number of training samples: 0


In [ ]:


# 1. Load the SRTM DEM dataset (30m resolution)
dem = ee.Image("USGS/SRTMGL1_003")

# 2. Define an area of interest (Modify coordinates for your region)
roi = ee.Geometry.Rectangle([-80.30, 32.55, -79.75, 33.00])

# 3. Clip DEM to the region of interest
dem_clipped = dem.clip(roi)

# 4. (Optional) Compute slope from the clipped DEM
slope = ee.Terrain.slope(dem_clipped)

# 5. Get min/max elevation and slope values for normalization
stats = dem_clipped.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi,
    scale=30,
    bestEffort=True
)
min_elev = stats.getNumber("elevation_min")
max_elev = stats.getNumber("elevation_max")

slope_stats = slope.reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=roi,
    scale=30,
    bestEffort=True
)
min_slope = slope_stats.getNumber("slope_min")
max_slope = slope_stats.getNumber("slope_max")

# 6. Normalize DEM and slope (convert to 0-1 scale)
dem_norm = dem_clipped.unitScale(min_elev, max_elev)
slope_norm = slope.unitScale(min_slope, max_slope)

# Combine DEM and slope into a single 2-band image (rename for clarity)
export_image = dem_norm.addBands(slope_norm).rename(["dem_norm", "slope_norm"])

# 7. Export the image to Google Drive as a GeoTIFF
task = ee.batch.Export.image.toDrive(
    image=export_image,
    description="DEM_Slope_Export",
    folder="EarthEngineExports",       # Change to your preferred Drive folder
    fileNamePrefix="dem_slope_norm",   # File name prefix for the exported TIFF
    region=roi.getInfo()["coordinates"],
    scale=30,
    crs="EPSG:4326",
    maxPixels=1e13
)

task.start()
print("Export started. Check the Tasks tab in the EE Code Editor or run task.status() to monitor.")



Export started. Check the Tasks tab in the EE Code Editor or run task.status() to monitor.


In [ ]:
# 7. Export the image to Google Drive
task = ee.batch.Export.image.toDrive(
    image=export_image,
    description="Normalized_DEM_Slope",
    folder="EarthEngineExports",  # Adjust to your desired Drive folder
    fileNamePrefix="dem_slope_norm",
    region=roi.getInfo()['coordinates'],  # or simply roi if you prefer
    scale=30,
    crs="EPSG:4326",
    maxPixels=1e13
)

task.start()
print("Export task started. Check the Tasks tab in GEE or run task.status().")


NameError: name 'export_image' is not defined

In [ ]:
# Create and train the classifier
classifier = ee.Classifier.smileRandomForest(
    numberOfTrees=50,
    seed=0
).train(
    features = training_sample,
    classProperty = 'class',
    inputProperties = train_bands
)

# Classify the entire image
classified = charleston_image.select(train_bands).classify(classifier)


In [ ]:
# Randomly split the FeatureCollection 70/30
withRandom = training_sample.randomColumn(columnName='random', seed=42)
split = 0.7
trainingFC = withRandom.filter(ee.Filter.lt('random', split))
testingFC  = withRandom.filter(ee.Filter.gte('random', split))

trained_classifier = ee.Classifier.smileRandomForest(50).train(
    features=trainingFC,
    classProperty='class',
    inputProperties=train_bands
)

test_classified = testingFC.classify(trained_classifier)

# Compute confusion matrix
confusion_matrix = test_classified.errorMatrix('class', 'classification')
print("Confusion Matrix:\n", confusion_matrix.getInfo())
print("Overall Accuracy:", confusion_matrix.accuracy().getInfo())


In [ ]:
# Create an interactive map
Map = geemap.Map(center=[32.8, -80.0], zoom=9)

# Define a palette for your classes (assuming 4 classes, for example)
class_palette = ['red', 'green', 'blue', 'yellow']

Map.addLayer(charleston_image,
             {'bands':['SR_B5_norm','SR_B4_norm','SR_B3_norm'], 'min':0, 'max':1},
             'Landsat (RGB)')

Map.addLayer(
    classified,
    {'min': 0, 'max': 3, 'palette': class_palette},
    'Land Cover Classification'
)

Map.addLayerControl()
Map
